# Decision trees for regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
A decision tree is like the game "twenty questions": "Is the size below 80? Yes → is the building older than 20 years? ...". In each **leaf**, the prediction is the **mean** $y$ of the samples that reached that leaf. The result is a **step** function.

> [A regression tree of depth 2. Each node is a question; value is the mean of that region.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/tree_reg_struct.png)

*A regression tree of depth 2. Each node is a question; value is the mean of that region.*

## How does the tree choose a question?
For every feature and every possible cut $t$, it splits the data into a left group ($x\le t$) and a right group and computes the **sum of squared errors**. The best cut gives the smallest value:

$$\text{SSE}=\sum_{i\in L}(y_i-\bar y_L)^2+\sum_{i\in R}(y_i-\bar y_R)^2$$

Then it repeats the same thing inside each group (**recursive splitting**) until a stopping rule is met.

## A worked example of one split
$x=[1,2,3,4]$ and $y=[1,1,5,6]$:

| Cut | Left group | Right group | SSE |
|---|---|---|---|
| $x\le1.5$ | [1] | [1, 5, 6] | $0+14=14$ |
| $x\le2.5$ | [1, 1] | [5, 6] | $0+0.5=0.5$ |
| $x\le3.5$ | [1, 1, 5] | [6] | $10.67+0=10.67$ |

The best cut is $x\le2.5$: the left side predicts 1 and the right side 5.5.



In [ ]:
import numpy as np

# Find the best cut by trying every possible threshold
x = np.array([1, 2, 3, 4]); y = np.array([1, 1, 5, 6])
for t in [1.5, 2.5, 3.5]:
    L, R = y[x <= t], y[x > t]
    sse = np.sum((L - L.mean()) ** 2) + np.sum((R - R.mean()) ** 2)
    print(f"x <= {t}: left {L}, right {R}, SSE = {sse:.2f}")



## Tree depth and overfitting
> [Depth 1: only two steps. Depth 3: good. No limit: passes through every point (overfitting).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/tree_reg_depth.png)

*Depth 1: only two steps. Depth 3: good. No limit: passes through every point (overfitting).*

[[lab:reg_tree]]

| Hyperparameter | Effect |
|---|---|
| `max_depth` | Maximum depth; lower is simpler |
| `min_samples_leaf` | Minimum samples per leaf; higher is smoother |
| `min_samples_split` | Minimum samples needed to split a node |
| `ccp_alpha` | Cost-complexity pruning; removes weak branches |

## Pros and cons
| Pros | Cons |
|---|---|
| Easy to understand and to draw | Unstable: a small data change gives a very different tree |
| Needs no scaling | Step-shaped output, not smooth |
| Finds non-linear relations and interactions by itself | Overfits quickly on its own |
| Numeric and categorical features | Does not extrapolate |

> The cure for an unstable tree is to combine many trees: random forests and boosting (next two lessons).



In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor, export_text

d = load_diabetes(as_frame=True)
for depth in [1, 2, 3, 4, 6, 10, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=depth, random_state=0), d.data, d.target, cv=5, scoring="r2")
    print(f"max_depth={depth}: R2 = {s.mean():.3f}")

tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(d.data, d.target)
print(export_text(tree, feature_names=list(d.data.columns)))

### Exercise

Build a DecisionTreeRegressor with min_samples_leaf=20 on diabetes and compare its CV R² with max_depth=None.

In [ ]:
# Your answer here